# Goldilocks pilot 02 — analysis
Computes representational movement, diversity and returnability from notebook 01 output. Add task-quality scores before drawing scientific conclusions.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn


In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

rows = [json.loads(line) for line in open(DATA_PATH, encoding='utf-8') if line.strip()]
df = pd.DataFrame(rows)
print(df.shape)
df.head()


In [ ]:
def vec(row, layer=-1):
    return np.asarray(row['layer_vectors'][layer], dtype=np.float32)

def cos(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else np.nan

def layer_velocity(row):
    x = np.asarray(row['layer_vectors'], dtype=np.float32)
    values = [1 - cos(x[i], x[i-1]) for i in range(1, len(x))]
    return float(np.nanmean(values))

df['velocity'] = df.apply(layer_velocity, axis=1)
df['quality'] = np.nan  # fill with exact or human/LLM-judge scores
df[['task_id','condition','strength','phase','velocity']].head()


In [ ]:
paired = []
keys = ['task_id', 'condition', 'strength', 'seed']
for key, group in df.groupby(keys):
    by_phase = {r['phase']: r for _, r in group.iterrows()}
    if 'baseline' not in by_phase:
        continue
    b = vec(by_phase['baseline'])
    p = vec(by_phase['perturbed']) if 'perturbed' in by_phase else b
    r = vec(by_phase['return']) if 'return' in by_phase else b
    paired.append({
        **dict(zip(keys, key)),
        'movement': 1 - cos(b, p),
        'returnability': cos(b, r),
        'retained_change': 1 - cos(b, r),
        'perturbation_to_return': 1 - cos(p, r),
        'baseline_velocity': float(by_phase['baseline']['velocity']),
        'perturbed_velocity': float(by_phase.get('perturbed', by_phase['baseline'])['velocity']),
        'return_velocity': float(by_phase.get('return', by_phase['baseline'])['velocity']),
    })
metrics = pd.DataFrame(paired)
metrics.head()


In [ ]:
def pairwise_diversity(group):
    vectors = np.stack([vec(row) for _, row in group.iterrows()])
    if len(vectors) < 2:
        return 0.0
    sim = cosine_similarity(vectors)
    upper = sim[np.triu_indices_from(sim, k=1)]
    return float(np.mean(1 - upper))

diversity = (
    df.groupby(['task_id','condition','strength','phase'])
      .apply(pairwise_diversity, include_groups=False)
      .rename('diversity')
      .reset_index()
)
summary = metrics.groupby(['condition','strength']).agg(
    movement=('movement','mean'),
    returnability=('returnability','mean'),
    retained_change=('retained_change','mean'),
    n=('seed','count')
).reset_index()
summary


## Required scoring before falsification
Add three independent scores per returned answer:

- task quality or exact correctness
- purpose integrity
- evidence fidelity

Then define epistemic gain as `quality_return - quality_baseline`. A Goldilocks effect requires moderate perturbation to outperform both none and strong perturbation while returnability and purpose integrity remain above preregistered thresholds.

In [ ]:
plt.figure(figsize=(8, 6))
for (condition, strength), g in metrics.groupby(['condition','strength']):
    plt.scatter(g['movement'], g['returnability'], label=f'{condition}:{strength}', alpha=0.7)
plt.xlabel('Representational movement')
plt.ylabel('Returnability')
plt.title('Goldilocks phase map — preliminary')
plt.legend(fontsize=8)
plt.grid(alpha=0.2)
plt.show()


In [ ]:
metrics.to_csv('/content/goldilocks_metrics.csv', index=False)
diversity.to_csv('/content/goldilocks_diversity.csv', index=False)
files.download('/content/goldilocks_metrics.csv')
